# Build the priors

Turns each downloaded prior into one of the two forms that [`pwpa.py`](pwpa.py) reads. PIE's five sources are all profiles.

- **profile**, `priors/profile/<source>.parquet`: one row per gene. The similarity of two genes is the dot product of their rows, so the rows are scaled first. A pair is covered when both genes have a row.
- **network**, `priors/network/<source>.parquet`: weighted gene pairs. A missing pair has similarity 0 and still counts as covered.

At fit time `pwpa.py` z-scores every prior over the training pairs, which puts all priors on a common scale.

For PIE's sources the similarity is the Pearson correlation of the two genes' DepMap gene effects, and the cosine of the two embeddings for the other four.
`ncbi_text` stores one vector per token, and a gene's vector is their mean.
Rows are indexed by human gene symbol, the key PIE looks genes up by, so the same files serve every dataset; the mouse datasets name each knockdown by its human orthologue.
As in PIE, a gene missing from a source is looked up again through the source's `aliases.yaml`.

In [ ]:
import io
import json
import os
import re
import sys
import tarfile
import zipfile

import numpy as np
import pandas as pd

sys.path.append("../utils")
from paths import data_dir

import pwpa

PRIORS = os.environ.get("PRIORS", "pie,tcell").split(",")  # which to build: PIE's five sources, the T-cell screens


In [ ]:
def standard_rows(table):
    """Profile rows from a numeric gene x feature table (screens, DepMap, expression): each row centred and scaled to
    unit length, so the dot product of two rows is their Pearson correlation. A missing value counts as the row mean."""
    return unit_rows(table.sub(table.mean(axis=1), axis=0).fillna(0))


def unit_rows(X):
    """Rows scaled to unit length, so the dot product of two rows is their cosine. A row of zeros carries no
    information, so its gene is dropped (left uncovered)."""
    norm = np.linalg.norm(X, axis=1)
    return X[norm > 0].div(norm[norm > 0], axis=0)

In [ ]:
raw = data_dir("priors", "raw")
for source in ["perturbation_text", "ncbi_text", "esm2", "string_space", "depmap_gene_effect"] if "pie" in PRIORS else []:
    meta = json.loads((raw / source / "meta.json").read_text())
    emb = np.load(raw / source / "embeddings.npy", mmap_mode="r")
    if meta["layout"] == "token":  # ncbi_text, 43 GB: read one gene's tokens at a time
        offsets = np.load(raw / source / "offsets.npy")
        vectors = np.stack([emb[a:b].mean(0, dtype=np.float64) for a, b in zip(offsets[:-1], offsets[1:])])
    else:
        vectors = np.asarray(emb, np.float64)
    table = pd.DataFrame(vectors, index=pd.Index(meta["keys"], name="gene")).rename(columns=str)

    # PIE uses an alias only when the name itself is missing and the alias target is present
    path = raw / source / "aliases.yaml"
    aliases = dict(line.split(": ") for line in path.read_text().splitlines()) if path.exists() else {}  # flat "a: b"
    extra = {a: t for a, t in aliases.items() if a not in table.index and t in table.index}
    table = pd.concat([table, table.loc[list(extra.values())].set_axis(pd.Index(list(extra), name="gene"))])

    rows = standard_rows(table) if source == "depmap_gene_effect" else unit_rows(table)
    assert np.allclose(np.linalg.norm(rows, axis=1), 1, atol=1e-5), source
    rows.astype(np.float32).to_parquet(data_dir("priors", "profile") / f"{source}.parquet")
    print(source, rows.shape)

## T-cell screens

Five genome-wide CRISPR screens in primary human T cells, used only by [`3.further_analysis`](../3.further_analysis/) next to PIE's five sources: they are the five screens the original PWPA model on Marson's T cells kept.
Three come from BioGRID ORCS (Carnevale 2022, T-cell proliferation; Shifrut 2018, CD8+ T-cell proliferation under an adenosine agonist; Schmidt 2022, IL-2 production, CRISPRi) and two from the papers' supplementary MAGeCK tables (Liu 2026, T-cell abundance in tumours; Chen 2025, FOXP3 induction).
A screen gives each gene one score: its log fold change (ZLFC for Carnevale), z-scored over the genes the screen tested.
That one number is the gene's row, so the dot product of two rows is the product of their scores: alike when losing either gene pushes the phenotype the same way.
A row of one number cannot be scaled to unit length like the embeddings above, which would leave only its sign; `pwpa.py` z-scores every prior over the training pairs anyway.
The rows go to `priors/tcell/profile/`, so the benchmarks, which read `priors/profile/`, keep PIE's five sources.


In [ ]:
ORCS = {"ORCS_1905": ("1905", 1), "ORCS_1110": ("1110", 4), "ORCS_2424": ("2424", 5)}  # screen id, its log fold change column
MAGECK = {  # zip, the xlsx inside, its sheet; the score is MAGeCK's pos|lfc
    "Liu_tumour_vs_input": ("liu2026_supplementary_tables.zip", "Supplementary_Table_1.xlsx", "Tumor vs Input_gene_summary"),
    "Chen_FOXP3": ("chen2025_supplementary_tables.zip", "Supplementary Table 1.xlsx", "MAGECK gene"),
}
NON_TARGETING = r"(?i)non.?targeting|ntctrl|^nt$|control"


def screen_rows(genes, scores):
    """One-column profile rows: the score z-scored over the genes the screen tested, the first of a duplicate kept."""
    s = pd.Series(pd.to_numeric(pd.Series(list(scores)), errors="coerce").values, pd.Index(genes, name="gene").astype(str).str.strip())
    s = s[s.notna() & ~s.index.str.contains(NON_TARGETING)]
    s = s[~s.index.duplicated()]
    return ((s - s.mean()) / s.std(ddof=0)).to_frame("score")


if "tcell" in PRIORS:
    raw, screens = data_dir("priors", "raw", "tcell_screens"), {}
    ids = {sid: (name, col) for name, (sid, col) in ORCS.items()}
    with tarfile.open(next(raw.glob("BIOGRID-ORCS-ALL-homo_sapiens-*.screens.tar.gz"))) as tar:
        for member in tar:
            sid = re.search(r"SCREEN_(\d+)-", member.name)
            if sid and sid.group(1) in ids:
                name, col = ids[sid.group(1)]
                t = pd.read_csv(tar.extractfile(member), sep="\t", dtype=str, na_values=["-"], keep_default_na=False)
                screens[name] = screen_rows(t["OFFICIAL_SYMBOL"], t[f"SCORE.{col}"])
                if len(screens) == len(ORCS):
                    break
    for name, (archive, table, sheet) in MAGECK.items():
        with zipfile.ZipFile(raw / archive) as z:
            xlsx = z.read(next(n for n in z.namelist() if n.endswith(table)))
        t = pd.read_excel(io.BytesIO(xlsx), sheet_name=sheet, header=None, engine="calamine")
        header = t.index[t[0] == "id"][0]
        t = t.iloc[header + 1 :].set_axis(t.loc[header].astype(str), axis=1)
        screens[name] = screen_rows(t["id"], t["pos|lfc"])
    assert len(screens) == 5
    for name, rows in screens.items():
        rows.astype(np.float32).to_parquet(data_dir("priors", "tcell", "profile") / f"{name}.parquet")
        print(name, rows.shape)


In [ ]:
priors = pwpa.load_priors(data_dir("priors"))
print(len(priors), "priors:", ", ".join(priors))
print(", ".join(pwpa.load_priors(data_dir("priors", "tcell"))), "(3.further_analysis only)")
